# Multinomial logit (MNL)

## Model

For decision maker $n$ and available alternative $j\in\mathcal{C}_n$,

$$
U_{nj}=V_{nj}+\varepsilon_{nj},\qquad
P_{nj}=
\frac{a_{nj}\exp(V_{nj})}
{\sum_{k\in\mathcal{C}_n}a_{nk}\exp(V_{nk})},
$$

where $a_{nj}$ is the availability indicator and the errors are independent
type-I extreme-value draws. This example estimates generic time and cost
coefficients, two alternative-specific constants, two income interactions, and
a robust covariance matrix:

$$
\begin{aligned}
V_{n,\mathrm{TRAIN}}&=\mathrm{ASC}_{T}
+\beta_t t_{nT}+\beta_c c_{nT}+\gamma_T z_n,\\
V_{n,\mathrm{SM}}&=\beta_t t_{nS}+\beta_c c_{nS},\\
V_{n,\mathrm{CAR}}&=\mathrm{ASC}_{C}
+\beta_t t_{nC}+\beta_c c_{nC}+\gamma_C z_n.
\end{aligned}
$$

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, MultinomialLogit, UtilitySpec
from torchdcm.datasets import make_swissmetro_like
# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Generate a Swissmetro-like sample and standardize income for interactions.
frame = make_swissmetro_like(n_obs=360, seed=7)
frame["income_std"] = (
    (frame["income"] - frame["income"].mean()) / frame["income"].std()
)
# Map wide columns, availability, and IDs into TorchDCM choice data.
data = ChoiceDataset.from_wide(
    frame,
    alternatives=["TRAIN", "SM", "CAR"],
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
        "income_std": {
            "TRAIN": "income_std",
            "SM": "income_std",
            "CAR": "income_std",
        },
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)

# Share time and cost effects across modes, with SM as the reference.
spec = UtilitySpec()
spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN")
    + Beta("B_TIME", init=-0.01) * "time"
    + Beta("B_COST", init=-0.10) * "cost"
    + Beta("B_INCOME_TRAIN", init=0.05) * "income_std",
)
spec.utility(
    "SM",
    Beta("B_TIME", init=-0.01) * "time"
    + Beta("B_COST", init=-0.10) * "cost",
)
spec.utility(
    "CAR",
    Beta("ASC_CAR")
    + Beta("B_TIME", init=-0.01) * "time"
    + Beta("B_COST", init=-0.10) * "cost"
    + Beta("B_INCOME_CAR", init=0.05) * "income_std",
)
print(f"Observations: {data.n_obs}; alternative rows: {data.n_rows}")


Observations: 360; alternative rows: 1080


In [3]:
# Fit the MNL and request a robust sandwich covariance matrix.
model = MultinomialLogit(spec, device=device, max_iter=80)
result = model.fit(data, cov_type="robust")
# Render the structured estimation report directly in the notebook.
display(HTML(result.report().to_html()))


Model family,MultinomialLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
